In [72]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [73]:
import sys
from pathlib import Path

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.append(str(root / "src" / "tabular"))

In [74]:
from gridworld import STATES, TERMINALS, ACTIONS, transitions, show

for a in range(4):
    print(ACTIONS[a], transitions((2, 0), a))

^ [(0.8, (1, 0), -0.04, False), (0.1, (2, 0), -0.04, False), (0.1, (2, 1), -0.04, False)]
> [(0.8, (2, 1), -0.04, False), (0.1, (1, 0), -0.04, False), (0.1, (2, 0), -0.04, False)]
v [(0.9, (2, 0), -0.04, False), (0.1, (2, 1), -0.04, False)]
< [(0.9, (2, 0), -0.04, False), (0.1, (1, 0), -0.04, False)]


In [75]:
def q_value(s, a, V, gamma=1.0):
    total = 0.0
    for p, ns, r, d in transitions(s, a):
        total += p * (r + gamma * V[ns])
    return total

In [76]:
V0 = {s: 0.0 for s in STATES}
print([round(q_value((2, 0), a, V0), 3) for a in range(4)])

[-0.04, -0.04, -0.04, -0.04]


In [77]:
V1 = {s: 0.0 for s in STATES}
V1[(1, 2)] = 0.66
print([round(q_value((0, 2), a, V1), 3) for a in range(4)])

[0.064, 0.858, 0.592, 0.026]


In [78]:
def value_iteration(gamma=1.0, theta=1e-6, verbose=False):
    V = {s: 0.0 for s in STATES}
    for sweep in range(1000):
        delta = 0.0
        for s in STATES:
            if s in TERMINALS:
                continue  # terminals are worth 0 by definition
            old = V[s]
            V[s] = max(
                q_value(s, a, V, gamma) for a in range(4)
            )  # max q_value over the 4 actions
            delta = max(delta, abs(V[s] - old))
        if verbose:
            print(f"sweep {sweep + 1}, delta={delta:.4f}")
            show(V=V)
        if delta < theta:
            print(f"converged in {sweep + 1} sweeps")
            break
    return V

V = value_iteration(verbose=True)

sweep 1, delta=0.7920
-0.04  -0.04  +0.79  +1.00 
-0.04    ##   +0.50  -1.00 
-0.04  -0.04  +0.35  +0.15 

sweep 2, delta=0.6256
-0.08  +0.59  +0.92  +1.00 
-0.08    ##   +0.65  -1.00 
-0.08  +0.24  +0.52  +0.29 

sweep 3, delta=0.4925
+0.41  +0.81  +0.95  +1.00 
+0.27    ##   +0.69  -1.00 
+0.19  +0.42  +0.58  +0.36 

sweep 4, delta=0.2846
+0.68  +0.88  +0.96  +1.00 
+0.56    ##   +0.70  -1.00 
+0.47  +0.51  +0.60  +0.38 

sweep 5, delta=0.1520
+0.79  +0.90  +0.96  +1.00 
+0.70    ##   +0.70  -1.00 
+0.62  +0.56  +0.61  +0.39 

sweep 6, delta=0.0692
+0.83  +0.91  +0.96  +1.00 
+0.76    ##   +0.70  -1.00 
+0.69  +0.62  +0.62  +0.40 

sweep 7, delta=0.0389
+0.84  +0.91  +0.96  +1.00 
+0.79    ##   +0.70  -1.00 
+0.72  +0.66  +0.63  +0.41 

sweep 8, delta=0.0190
+0.85  +0.91  +0.96  +1.00 
+0.80    ##   +0.70  -1.00 
+0.74  +0.68  +0.64  +0.41 

sweep 9, delta=0.0084
+0.85  +0.91  +0.96  +1.00 
+0.80    ##   +0.70  -1.00 
+0.74  +0.69  +0.65  +0.42 

sweep 10, delta=0.0036
+0.85  +0.91  

In [79]:
def greedy_policy(V, gamma=1.0):
    pi = {}
    for s in STATES:
        if s in TERMINALS:
            continue
        pi[s] = max(range(4), key=lambda a: q_value(s, a, V, gamma))
    return pi

show(pi=greedy_policy(V))


  >      >      >    +1.00 
  ^      ##     ^    -1.00 
  ^      <      <      <   



In [80]:
def policy_evaluation(pi, gamma=1.0, theta=1e-6):
    V = {s: 0.0 for s in STATES}
    for _ in range(1000):
        delta = 0.0
        for s in STATES:
            if s in TERMINALS:
                continue
            old = V[s]
            V[s] = q_value(s, pi[s], V, gamma)
            delta = max(delta, abs(V[s] - old))
        if delta < theta:
            break
    return V


def policy_iteration(gamma=1.0):
    pi = {s: 0 for s in STATES if s not in TERMINALS}   # start: everyone goes up
    V = None
    for i in range(100):
        V = policy_evaluation(pi, gamma)
        new_pi = greedy_policy(V, gamma)
        if new_pi == pi:
            print(f"converged in {i + 1} iterations")
            break
        pi = new_pi
    return V, pi


V_pi, pi = policy_iteration()
show(V=V_pi)
show(pi=pi)


converged in 5 iterations
+0.85  +0.91  +0.96  +1.00 
+0.80    ##   +0.70  -1.00 
+0.75  +0.70  +0.65  +0.43 

  >      >      >    +1.00 
  ^      ##     ^    -1.00 
  ^      <      <      <   

